# 第20章　データ管理の実装 ― 対応表とパイプライン**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## 20.1　対応表を中心に据える

In [ ]:
import pandas as pddf = pd.read_csv("dataset_mapping.csv")# 列の例: case_id, patient_id, image_path, label_path, disease, source, splitprint(df["disease"].value_counts())          # クラスの内訳（不均衡の把握）print(df["source"].value_counts())           # 施設・由来の偏り（バイアスの把握）

## 20.2　患者単位で、層化して分割する

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold# 患者単位(groups)で、疾患(y)を層化して5分割sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)X, y, groups = df.index, df["disease"], df["patient_id"]for fold, (tr, va) in enumerate(sgkf.split(X, y, groups)):    df.loc[df.index[va], "fold"] = fold# 検証: 同じ患者が複数foldにまたがっていないかassert df.groupby("patient_id")["fold"].nunique().max() == 1# 分割にも版を付ける。データの中身が同じでも、分け方が変われば別の実験になる（20.4のデータ版とは別に持つ）import hashlib, jsonsplit_version = hashlib.sha256(json.dumps(    sorted(zip(df["case_id"].astype(str), df["fold"].astype(int))), ensure_ascii=False).encode("utf-8")).hexdigest()[:12]print("split version:", split_version)

## 分割を目で確かめる ― foldの偏りを可視化する

In [ ]:
rows = []for f in range(5):    sub = df[df["fold"] == f]    rows.append({        "fold": f,        "患者数": sub["patient_id"].nunique(),        "画像数": len(sub),        "陽性率": round((sub["disease"] != "正常").mean(), 3),   # 文字列列なので、まず陽性かどうかに直す    })summary = pd.DataFrame(rows)print(summary.to_string(index=False))# 全患者が、どこか一つのfoldにだけ属することを機械的に再確認assert df.groupby("patient_id")["fold"].nunique().max() == 1, "患者リークあり"

In [ ]:
import matplotlib.pyplot as pltsummary.plot.bar(x="fold", y="陽性率", legend=False, ylim=(0, 0.5))plt.axhline((df["disease"] != "正常").mean(), ls="--")  # 全体の陽性率を基準線にplt.ylabel("positive rate"); plt.tight_layout(); plt.savefig("fold_balance.png")

## 20.3　品質を検証する

In [ ]:
from pathlib import Pathassert df["case_id"].is_unique, "症例IDに重複あり"# 除外理由の列が無ければ空で用意する。理由は上書きせず、追記する（複数理由を残す）if "exclude_reason" not in df.columns:    df["exclude_reason"] = pd.NAdef add_reason(mask, reason):    cur = df.loc[mask, "exclude_reason"]    df.loc[mask, "exclude_reason"] = cur.where(cur.notna(), "").astype(str).str.cat(        [reason] * int(mask.sum()), sep=";").str.strip(";")img_missing = ~df["image_path"].apply(lambda p: Path(p).exists())lbl_missing = ~df["label_path"].apply(lambda p: Path(p).exists())add_reason(img_missing, "画像ファイル欠損")add_reason(lbl_missing, "ラベルファイル欠損")# 検出して警告するだけでは学習用に残ってしまう。理由を書き込んでから隔離するprint(f"画像欠損 {int(img_missing.sum())}件 / ラベル欠損 {int(lbl_missing.sum())}件 を除外理由に記録")excluded = df[df["exclude_reason"].notna()]excluded.to_csv("excluded_cases.csv", index=False)df = df[df["exclude_reason"].isna()]             # 学習には除外後を使う

## 20.4　版を刻む

In [ ]:
import hashlib, jsondef file_hash(path, buf=1 << 20):                    # ファイルの中身そのものを要約する    h = hashlib.sha256()    with open(path, "rb") as f:        for chunk in iter(lambda: f.read(buf), b""):            h.update(chunk)    return h.hexdigest()def dataset_hash(df, schema_version, preprocess_version):    """case_idだけを繋いでも版にならない。画像とラベルの中身、除外条件、    スキーマと前処理の版まで含めて、はじめて「このデータで学習した」と言える。"""    cases = [{"case_id": str(r.case_id), "patient_id": str(r.patient_id),              "image": file_hash(r.image_path), "label": file_hash(r.label_path),              "exclude_reason": None if pd.isna(r.exclude_reason) else str(r.exclude_reason)}             for r in df.itertuples()]    ids = [c["case_id"] for c in cases]    assert len(ids) == len(set(ids)), "case_id が重複している"   # 一意でなければ版にならない    cases.sort(key=lambda c: c["case_id"])       # 辞書同士は比較できないので、キーを指定して並べる    manifest = {        "schema_version": schema_version,        "preprocess_version": preprocess_version,        "cases": cases,    }    # 区切りのないID連結は衝突する（["1","23"] と ["12","3"] はどちらも "123"）。    # キー順を固定したJSONへ直列化してから要約する。    blob = json.dumps(manifest, sort_keys=True, ensure_ascii=False).encode("utf-8")    return hashlib.sha256(blob).hexdigest()[:12]version = dataset_hash(df, schema_version="v3", preprocess_version="resample-1.5mm-v2")df.to_csv(f"dataset_v_{version}.csv", index=False)   # 版を刻んで保存print(f"dataset version: {version}")# 分割（train/val/test の割り当て）は別の版として持つ。データの中身が同じでも、# 分け方を変えれば別の実験になる。学習の成果物には、データ版・分割版・前処理版の# 3つを一緒に保存する（第22章のチェックポイント、第24章の推論用成果物）。

## ミニプロジェクト ― ラベル分布を可視化し、不均衡と施設バイアスを一目で暴く

In [ ]:
import pandas as pd, numpy as npimport matplotlib.pyplot as plt# --- 対応表（無ければ模擬データで代用） ---try:    df = pd.read_csv("dataset_mapping.csv")except FileNotFoundError:    rng = np.random.default_rng(0)    n = 600    df = pd.DataFrame({        "disease": rng.choice(["正常", "良性", "悪性"], n, p=[0.80, 0.15, 0.05]),        "source":  rng.choice(["A病院", "B病院"], n, p=[0.7, 0.3]),    })# --- ① クラスの内訳と不均衡比 ---counts = df["disease"].value_counts()imbalance = counts.max() / counts.min()          # 多数派は少数派の何倍かprint(counts)print(f"不均衡比（最多/最少）= {imbalance:.1f} 倍")print(f"最少クラスの枚数 = {counts.min()} 枚  ← 学習が届く上限はここで決まる")# --- ② 施設×疾患のクロス集計（バイアスの発見） ---cross = pd.crosstab(df["source"], df["disease"], normalize="index")print("\n施設ごとの疾患割合（行で正規化）:\n", cross.round(2))# --- ③ 図にして保存する ---# 目盛り・凡例はデータの値がそのまま出る。既定フォントは日本語を持たないので英語に貼り替えるja2en = {"正常": "normal", "良性": "benign", "悪性": "malignant",         "A病院": "site A", "B病院": "site B"}counts_en = counts.rename(index=ja2en)cross_en  = cross.rename(index=ja2en, columns=ja2en)fig, ax = plt.subplots(1, 2, figsize=(12, 4))counts_en.plot.bar(ax=ax[0], color="steelblue")ax[0].set_title(f"class distribution (imbalance {imbalance:.1f}x)")   # 図中は英語ax[0].set_ylabel("count")cross_en.plot.bar(stacked=True, ax=ax[1])ax[1].set_title("disease mix by site"); ax[1].set_ylabel("share")   # 割合の内訳plt.tight_layout(); plt.savefig("label_distribution.png", dpi=150)print("\nlabel_distribution.png を保存しました")